# 01 — Audit Dataset

Notebook ini adalah **gerbang** sebelum retrain (spec §5, §13 langkah [2]).
Tujuannya murni verifikasi empirik — tidak mengasumsikan apa pun dari
eksperimen lama:

1. Struktur folder & pola nama file → apakah ada indikasi label region
2. **Deteksi kebocoran antar split** (perceptual hash) — ini yang paling kritis
3. Distribusi kelas per split
4. File korup → **dilaporkan ke CSV, TIDAK dihapus** (beda dari `deep_clean_images`
   di eksperimen lama yang menghapus langsung — lihat spec §2.2 R2)
5. Statistik dimensi/channel/ukuran gambar
6. `audit_report.json` — sumber tunggal hasil audit, dipakai gerbang keputusan

**Read-only.** Tidak ada sel di notebook ini yang mengubah/menghapus file dataset.

Ganti `DATASET_ROOT` di sel berikutnya kalau dataset yang dipakai berubah
(mis. dataset baru dari dosen) — seluruh sel setelahnya generik, tidak
mengasumsikan nama/struktur dataset tertentu.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.0 MB/s eta 0:00:00


In [3]:
import json
import datetime
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm

# ============================================================
# SATU-SATUNYA BARIS YANG PERLU DIGANTI KALAU DATASET BERUBAH
# ============================================================
DATASET_ROOT = Path('/content/drive/MyDrive/project-fracture/dataset/Bone_Fracture_Dataset')

OUTPUT_DIR = Path('/content/audit_output')
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
LEAKAGE_THRESHOLD_PCT = 1.0  # per spec §5 — di atas ini, split wajib dibuat ulang

assert DATASET_ROOT.exists(), f"DATASET_ROOT tidak ditemukan: {DATASET_ROOT}"
print("Dataset root:", DATASET_ROOT)

Dataset root: /content/drive/MyDrive/project-fracture/dataset/Bone_Fracture_Dataset


## 1. Struktur folder & split — dideteksi otomatis, tidak diasumsikan

In [4]:
# Split dideteksi dari isi folder, BUKAN di-hardcode ['train','val','test'] —
# supaya audit ini tetap benar walau dataset baru punya nama split berbeda
# (mis. 'valid' bukan 'val').
discovered_splits = sorted([d.name for d in DATASET_ROOT.iterdir() if d.is_dir()])
print("Split terdeteksi:", discovered_splits)

def discover_structure(root, splits):
    structure = {}
    for split in splits:
        split_dir = root / split
        classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
        structure[split] = {}
        for c in classes:
            files = [f for f in (split_dir / c).iterdir() if f.suffix.lower() in IMAGE_EXTS]
            structure[split][c] = len(files)
    return structure

structure = discover_structure(DATASET_ROOT, discovered_splits)
print(json.dumps(structure, indent=2))

Split terdeteksi: ['test', 'train', 'val']
{
  "test": {
    "fractured": 238,
    "not fractured": 268
  },
  "train": {
    "fractured": 4606,
    "not fractured": 4640
  },
  "val": {
    "fractured": 337,
    "not fractured": 492
  }
}


## 2. Indikasi label region di nama file

Menentukan apakah multi-task (binary + region) memungkinkan dengan dataset
ini. **Keputusan yang sudah dikunci (brainstorming 2026-08-12): kalau tidak
ada indikasi region, TETAP biner** — bukan alasan pindah dataset.

In [5]:
REGION_TOKENS = [
    'wrist', 'hand', 'elbow', 'shoulder', 'forearm', 'humerus', 'finger',
    'hip', 'knee', 'ankle', 'foot', 'femur', 'tibia', 'fibula', 'clavicle',
    'pergelangan', 'tangan', 'siku', 'bahu', 'lengan', 'jari', 'pinggul',
    'lutut', 'kaki',
]

sample_files = []
for split, classes in structure.items():
    for c in classes:
        split_dir = DATASET_ROOT / split / c
        sample_files.extend(list(split_dir.iterdir())[:25])

region_hits = Counter()
for f in sample_files:
    name = f.name.lower()
    for token in REGION_TOKENS:
        if token in name:
            region_hits[token] += 1

has_region_labels = len(region_hits) > 0
print("Contoh nama file:", [f.name for f in sample_files[:5]])
print("Token region terdeteksi:", dict(region_hits) if region_hits else "TIDAK ADA")
print()
print("Kesimpulan: dataset", "PUNYA" if has_region_labels else "TIDAK PUNYA",
      "indikasi label region di nama file.")
if not has_region_labels:
    print("-> Scope TETAP biner (fractured/not_fractured), sesuai keputusan yang sudah dikunci.")

Contoh nama file: ['00001.png', '0012.png', '000151898.png', '0010.png', '0005156.png']
Token region terdeteksi: TIDAK ADA

Kesimpulan: dataset TIDAK PUNYA indikasi label region di nama file.
-> Scope TETAP biner (fractured/not_fractured), sesuai keputusan yang sudah dikunci.


## 3. File korup — dilaporkan, TIDAK dihapus

In [6]:
all_files = []
for split, classes in structure.items():
    for c in classes:
        split_dir = DATASET_ROOT / split / c
        for f in split_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXTS:
                all_files.append((split, c, f))

print(f"Total file ditemukan: {len(all_files)}")

bad_files = []
for split, c, f in tqdm(all_files, desc="Cek file korup"):
    try:
        with Image.open(f) as img:
            img.load()
    except Exception as e:
        bad_files.append({'split': split, 'class': c, 'path': str(f), 'error': str(e)})

print(f"\nFile korup ditemukan: {len(bad_files)} dari {len(all_files)}")
if bad_files:
    pd.DataFrame(bad_files).to_csv(OUTPUT_DIR / 'corrupt_files.csv', index=False)
    print("Daftar disimpan ke corrupt_files.csv — TIDAK dihapus otomatis dari dataset.")
    print("Keputusan hapus/tidak dilakukan manual, di luar notebook ini.")

Total file ditemukan: 10581


Cek file korup:   0%|          | 0/10581 [00:00<?, ?it/s]


File korup ditemukan: 18 dari 10581
Daftar disimpan ke corrupt_files.csv — TIDAK dihapus otomatis dari dataset.
Keputusan hapus/tidak dilakukan manual, di luar notebook ini.


## 4. Distribusi kelas per split

In [7]:
dist_rows = []
for split, classes in structure.items():
    total = sum(classes.values())
    for c, n in classes.items():
        dist_rows.append({'split': split, 'class': c, 'count': n, 'pct': round(n / total * 100, 2) if total else 0})

dist_df = pd.DataFrame(dist_rows)
display(dist_df.pivot(index='class', columns='split', values='count'))
display(dist_df.pivot(index='class', columns='split', values='pct'))

split,test,train,val
class,,,
fractured,238,4606,337
not fractured,268,4640,492


split,test,train,val
class,,,
fractured,47.04,49.82,40.65
not fractured,52.96,50.18,59.35


## 5. Statistik gambar — dimensi, channel, ukuran file

In [8]:
bad_paths = {b['path'] for b in bad_files}
stats_rows = []
for split, c, f in tqdm(all_files, desc="Kumpulkan statistik gambar"):
    if str(f) in bad_paths:
        continue
    try:
        with Image.open(f) as img:
            stats_rows.append({
                'split': split, 'class': c,
                'width': img.width, 'height': img.height,
                'mode': img.mode, 'size_kb': round(f.stat().st_size / 1024, 1),
            })
    except Exception:
        continue

stats_df = pd.DataFrame(stats_rows)
print("Dimensi & ukuran file per split:")
display(stats_df.groupby('split')[['width', 'height', 'size_kb']].describe().T)
print("\nDistribusi mode warna (RGB/L/dll) per split:")
display(stats_df.groupby(['split', 'mode']).size())

Kumpulkan statistik gambar:   0%|          | 0/10581 [00:00<?, ?it/s]

Dimensi & ukuran file per split:


split                 test        train          val
width   count   500.000000  9240.000000   823.000000
        mean    500.068000   277.906169   451.047388
        std     633.747153   269.949759   566.850533
        min     224.000000   100.000000   100.000000
        25%     224.000000   224.000000   224.000000
        50%     224.000000   224.000000   224.000000
        75%     224.000000   224.000000   249.000000
        max    2460.000000  4232.000000  3000.000000
height  count   500.000000  9240.000000   823.000000
        mean    585.036000   296.120455   516.258809
        std     807.937367   339.687794   724.582215
        min     224.000000   100.000000   100.000000
        25%     224.000000   224.000000   224.000000
        50%     224.000000   224.000000   224.000000
        75%     224.000000   224.000000   277.000000
        max    2970.000000  5823.000000  4403.000000
size_kb count   500.000000  9240.000000   823.000000
        mean    184.527800    32.799859   134.552734
        std     513.105796   156.660223   469.162680
        min       6.700000     1.100000     1.100000
        25%       9.600000     9.400000     9.200000
        50%      10.900000    10.600000    10.800000
        75%      12.700000    11.900000    13.600000
        max    3217.300000  3217.300000  6281.300000


Distribusi mode warna (RGB/L/dll) per split:


split  mode
test   L         89
       RGB      411
train  L        327
       P          6
       RGB     8894
       RGBA      13
val    L        119
       P          3
       RGB      694
       RGBA       7
dtype: int64

## 6. Deteksi kebocoran antar split (perceptual hash) — PALING KRITIS

Ini yang menjawab risiko di spec §2.6: apakah pembagian train/val/test
sungguhan bebas duplikat, atau ada gambar (identik/hasil augmentasi) yang
"bocor" dari satu split ke split lain — yang kalau terjadi, membuat semua
angka test menjadi terlalu optimis.

In [9]:
hash_map = defaultdict(list)  # phash (str) -> [{'split','class','path'}, ...]
hash_errors = 0

for split, c, f in tqdm(all_files, desc="Hitung perceptual hash"):
    if str(f) in bad_paths:
        continue
    try:
        with Image.open(f) as img:
            h = imagehash.phash(img)
        hash_map[str(h)].append({'split': split, 'class': c, 'path': str(f)})
    except Exception:
        hash_errors += 1

total_images = sum(len(v) for v in hash_map.values())
print(f"Gambar valid di-hash: {total_images}  |  error hashing: {hash_errors}")
print(f"Jumlah hash unik: {len(hash_map)}")

Hitung perceptual hash:   0%|          | 0/10581 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Gambar valid di-hash: 10563  |  error hashing: 0
Jumlah hash unik: 3370


In [10]:
# Kebocoran LINTAS SPLIT — hash yang sama muncul di lebih dari satu split
leaked = {h: entries for h, entries in hash_map.items() if len({e['split'] for e in entries}) > 1}
total_leaked_images = sum(len(v) for v in leaked.values())
leakage_pct = (total_leaked_images / total_images * 100) if total_images else 0.0

print(f"Grup hash yang bocor lintas split : {len(leaked)}")
print(f"Total gambar terlibat kebocoran   : {total_leaked_images} ({leakage_pct:.3f}% dari seluruh dataset)")

if leaked:
    leak_rows = [{'hash': h, **e} for h, entries in leaked.items() for e in entries]
    pd.DataFrame(leak_rows).to_csv(OUTPUT_DIR / 'leakage_report.csv', index=False)
    print("Detail lengkap (hash + path tiap gambar yang terlibat) -> leakage_report.csv")

    print("\nContoh 5 grup pertama:")
    for h, entries in list(leaked.items())[:5]:
        splits_involved = sorted({e['split'] for e in entries})
        print(f"  hash={h}  splits={splits_involved}  n_gambar={len(entries)}")

# Duplikat DALAM split yang sama (bukan kebocoran, tapi tetap dicatat —
# duplikat berlebihan di train saja bisa bias distribusi, bukan leakage)
intra_split_dupes = {
    h: entries for h, entries in hash_map.items()
    if len(entries) > 1 and len({e['split'] for e in entries}) == 1
}
print(f"\nGrup duplikat DALAM split yang sama (bukan leakage): {len(intra_split_dupes)}")

Grup hash yang bocor lintas split : 837
Total gambar terlibat kebocoran   : 3680 (34.839% dari seluruh dataset)
Detail lengkap (hash + path tiap gambar yang terlibat) -> leakage_report.csv

Contoh 5 grup pertama:
  hash=953be51eaa16d413  splits=['test', 'train', 'val']  n_gambar=13
  hash=c1c97e1d0f313876  splits=['test', 'train', 'val']  n_gambar=12
  hash=c7279c42b759a552  splits=['test', 'train', 'val']  n_gambar=15
  hash=daf6097975056d28  splits=['test', 'train', 'val']  n_gambar=12
  hash=b038ce30cf71cd66  splits=['test', 'train', 'val']  n_gambar=6

Grup duplikat DALAM split yang sama (bukan leakage): 2220


In [11]:
print("=" * 64)
print("GERBANG KEPUTUSAN — spec §5")
print("=" * 64)
gate_passed = leakage_pct <= LEAKAGE_THRESHOLD_PCT
if gate_passed:
    print(f"LOLOS — kebocoran {leakage_pct:.3f}% <= ambang {LEAKAGE_THRESHOLD_PCT}%.")
    print("   Split boleh dipakai apa adanya, lanjut ke tahap retrain (spec §6).")
else:
    print(f"GAGAL — kebocoran {leakage_pct:.3f}% > ambang {LEAKAGE_THRESHOLD_PCT}%.")
    print("   Split WAJIB dibuat ulang secara deterministik (seed tetap) sebelum")
    print("   training dimulai — lihat leakage_report.csv untuk detail gambar yang bocor.")

GERBANG KEPUTUSAN — spec §5
GAGAL — kebocoran 34.839% > ambang 1.0%.
   Split WAJIB dibuat ulang secara deterministik (seed tetap) sebelum
   training dimulai — lihat leakage_report.csv untuk detail gambar yang bocor.


## 7. Simpan `audit_report.json` — sumber tunggal hasil audit

In [12]:
audit_report = {
    'generated_at': datetime.datetime.now().isoformat(),
    'dataset_root': str(DATASET_ROOT),
    'splits_detected': discovered_splits,
    'structure': structure,
    'class_distribution': dist_rows,
    'total_images_scanned': len(all_files),
    'corrupt_files_count': len(bad_files),
    'has_region_label_hint': has_region_labels,
    'region_tokens_found': dict(region_hits),
    'leakage': {
        'unique_hashes': len(hash_map),
        'leaked_groups': len(leaked),
        'leaked_images': total_leaked_images,
        'leakage_pct': round(leakage_pct, 4),
        'threshold_pct': LEAKAGE_THRESHOLD_PCT,
        'gate_passed': gate_passed,
    },
    'intra_split_duplicate_groups': len(intra_split_dupes),
}

with open(OUTPUT_DIR / 'audit_report.json', 'w') as fh:
    json.dump(audit_report, fh, indent=2)

print("Tersimpan:", OUTPUT_DIR / 'audit_report.json')
print()
print(json.dumps(audit_report, indent=2)[:2000], "...")

Tersimpan: /content/audit_output/audit_report.json

{
  "generated_at": "2026-08-14T09:23:00.641594",
  "dataset_root": "/content/drive/MyDrive/project-fracture/dataset/Bone_Fracture_Dataset",
  "splits_detected": [
    "test",
    "train",
    "val"
  ],
  "structure": {
    "test": {
      "fractured": 238,
      "not fractured": 268
    },
    "train": {
      "fractured": 4606,
      "not fractured": 4640
    },
    "val": {
      "fractured": 337,
      "not fractured": 492
    }
  },
  "class_distribution": [
    {
      "split": "test",
      "class": "fractured",
      "count": 238,
      "pct": 47.04
    },
    {
      "split": "test",
      "class": "not fractured",
      "count": 268,
      "pct": 52.96
    },
    {
      "split": "train",
      "class": "fractured",
      "count": 4606,
      "pct": 49.82
    },
    {
      "split": "train",
      "class": "not fractured",
      "count": 4640,
      "pct": 50.18
    },
    {
      "split": "val",
      "class": "fractured",

## 8. Unduh hasil

Unduh `audit_report.json` (dan `corrupt_files.csv` / `leakage_report.csv`
kalau ada), lalu taruh di `results/audit_report.json` pada repo lokal —
ini yang jadi acuan keputusan lanjut/tidaknya ke tahap retrain.

In [13]:
from google.colab import files

files.download(str(OUTPUT_DIR / 'audit_report.json'))
if bad_files:
    files.download(str(OUTPUT_DIR / 'corrupt_files.csv'))
if leaked:
    files.download(str(OUTPUT_DIR / 'leakage_report.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>